# Практична робота № 3. Очищення та перетворення даних
**Виконав:** Мельниківський Андрій, група КН-41

**Набір даних:** `sales.csv` — продажі кав'ярні (300 транзакцій, 6 колонок).

## Крок 1. Підготовка середовища та завантаження даних

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Підключення Google Диска (у Colab). Поза Colab — використовується локальна папка.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/'
except ImportError:
    BASE = ''
    print('Colab не виявлено — використовується локальна папка')
PATH = BASE + 'sales.csv'
print('Шлях до файлу:', PATH)

Colab не виявлено — використовується локальна папка
Шлях до файлу: sales.csv


In [3]:
df = pd.read_csv(PATH)
df.head()

,date,datetime,cash_type,card,money,coffee_name
0,2025-03-01,2025-03-01 09:27:40.117,card,ANON-0000-0000-0001,35.99,Cappuccino
1,2025-03-01,2025-03-01 09:40:11.474,card,ANON-0000-0000-0002,35.99,Cocoa
2,2025-03-01,2025-03-01 10:12:26.345,cash,NaN,24.00,Espresso
3,2025-03-01,2025-03-01 10:38:22.609,card,ANON-0000-0000-0004,38.70,Latte
4,2025-03-01,2025-03-01 11:33:02.443,card,ANON-0000-0000-0005,33.80,Americano with Milk


Колонка `datetime` поки має текстовий тип — перетворимо її в кроці 5.

## Крок 2. Виявлення та обробка пропущених значень
Створюємо штучний DataFrame з пропусками.

In [4]:
# Синтетичні дані
data_messy = {
    'name': ['Anna', None, 'Bohdan', 'Olena', None],
    'age': [20, 21, None, 22, 24],
    'score': [85.5, 90.0, 78.3, None, 92.1]
}
df_messy = pd.DataFrame(data_messy)
df_messy

,name,age,score
0,Anna,20.0,85.5
1,None,21.0,90.0
2,Bohdan,NaN,78.3
3,Olena,22.0,NaN
4,None,24.0,92.1


In [5]:
print(df_messy.isnull().sum())  # кількість пропусків у кожній колонці

name     2
age      1
score    1
dtype: int64


In [6]:
# Видалення рядків з пропусками у колонці 'name'
df_dropped = df_messy.dropna(subset=['name'])  # видаляє рядки, де в name пропуск
print(df_dropped)

# Заповнення пропусків у колонці 'age' середнім значенням
mean_age = df_messy['age'].mean()  # середнє, ігноруючи NaN
df_filled = df_messy.copy()
df_filled['age'] = df_filled['age'].fillna(mean_age)  # заміна пропусків середнім
print(df_filled)

     name   age  score
0    Anna  20.0   85.5
2  Bohdan   NaN   78.3
3   Olena  22.0    NaN
     name    age  score
0    Anna  20.00   85.5
1    None  21.00   90.0
2  Bohdan  21.75   78.3
3   Olena  22.00    NaN
4    None  24.00   92.1


Перший блок залишає лише рядки, де є `name` (три рядки). Другий замінює пропуск віку середнім (21.75).

## Крок 3. Виявлення та видалення дублікатів

In [7]:
data_dup = {
    'id': [1, 2, 3, 2, 3],
    'value': ['A', 'B', 'C', 'B', 'C']
}
df_dup = pd.DataFrame(data_dup)
df_dup

,id,value
0,1,A
1,2,B
2,3,C
3,2,B
4,3,C


In [8]:
print(df_dup.duplicated())
print('Кількість дублікатів:', df_dup.duplicated().sum())

0    False
1    False
2    False
3     True
4     True
dtype: bool
Кількість дублікатів: 2


In [9]:
df_unique = df_dup.drop_duplicates()  # залишає перше входження кожного рядка
print(df_unique)

   id value
0   1     A
1   2     B
2   3     C


## Крок 4. Виявлення та обробка викидів (метод IQR)

In [10]:
Q1 = df['money'].quantile(0.25)
Q3 = df['money'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Q1 = {Q1}, Q3 = {Q3}, IQR = {IQR}")
print(f"Нижня межа: {lower}, Верхня межа: {upper}")

Q1 = 28.9, Q3 = 38.7, IQR = 9.800000000000004
Нижня межа: 14.199999999999992, Верхня межа: 53.400000000000006


In [11]:
outliers = df[(df['money'] < lower) | (df['money'] > upper)]
print(f"Знайдено викидів: {len(outliers)}")

Знайдено викидів: 0


Викидів немає, тож видаляти нічого. Якби вони були, їх можна було б **обмежити** граничними значеннями (`clip`), а не видаляти:

In [12]:
money_clipped = df['money'].clip(lower=lower, upper=upper)
print('Значення змінено:', (money_clipped != df['money']).sum(), 'шт.')

Значення змінено: 0 шт.


## Крок 5. Перетворення типів даних

In [13]:
print(df.dtypes, '\n')
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
print(df.dtypes)

date            object
datetime        object
cash_type       object
card            object
money          float64
coffee_name     object
dtype: object 

date                   object
datetime       datetime64[ns]
cash_type              object
card                   object
money                 float64
coffee_name            object
dtype: object


## Крок 6. Обробка рядкових даних

In [14]:
df_text = pd.DataFrame({
    'city': ['  Kyiv  ', 'LVIV', 'Odesa ', 'KHARKIV', ' kyiv '],
    'phone': ['(044) 123-45-67', '(032) 234-56-78', '(048) 345-67-89', '(057) 456-78-90', '(044) 123-45-67']
})
df_text

,city,phone
0,Kyiv,(044) 123-45-67
1,LVIV,(032) 234-56-78
2,Odesa,(048) 345-67-89
3,KHARKIV,(057) 456-78-90
4,kyiv,(044) 123-45-67


In [15]:
df_text['city'] = df_text['city'].str.strip().str.lower()
df_text['phone'] = df_text['phone'].str.replace(r'\(|\)|-', '', regex=True)
print(df_text)

      city        phone
0     kyiv  044 1234567
1     lviv  032 2345678
2    odesa  048 3456789
3  kharkiv  057 4567890
4     kyiv  044 1234567


Тепер `' Kyiv '` і `' kyiv '` — однакове значення `kyiv`; номери телефонів без дужок і дефісів. Це запобігає помилкам під час групування.

## Крок 7. Створення нових ознак (feature engineering)

In [16]:
df['date_only'] = df['datetime'].dt.date
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek
df['month'] = df['datetime'].dt.month
print(df[['datetime', 'date_only', 'hour', 'day_of_week', 'month']].head())

                 datetime   date_only  hour  day_of_week  month
0 2025-03-01 09:27:40.117  2025-03-01     9            5      3
1 2025-03-01 09:40:11.474  2025-03-01     9            5      3
2 2025-03-01 10:12:26.345  2025-03-01    10            5      3
3 2025-03-01 10:38:22.609  2025-03-01    10            5      3
4 2025-03-01 11:33:02.443  2025-03-01    11            5      3


## Крок 8. One-hot encoding категоріальних змінних

In [17]:
df_encoded = pd.get_dummies(df, columns=['coffee_name'])
print(df_encoded.head())

         date                datetime cash_type                 card  money  \
0  2025-03-01 2025-03-01 09:27:40.117      card  ANON-0000-0000-0001  35.99   
1  2025-03-01 2025-03-01 09:40:11.474      card  ANON-0000-0000-0002  35.99   
2  2025-03-01 2025-03-01 10:12:26.345      cash                  NaN  24.00   
3  2025-03-01 2025-03-01 10:38:22.609      card  ANON-0000-0000-0004  38.70   
4  2025-03-01 2025-03-01 11:33:02.443      card  ANON-0000-0000-0005  33.80   

    date_only  hour  day_of_week  month  coffee_name_Americano  \
0  2025-03-01     9            5      3                  False   
1  2025-03-01     9            5      3                  False   
2  2025-03-01    10            5      3                  False   
3  2025-03-01    10            5      3                  False   
4  2025-03-01    11            5      3                  False   

   coffee_name_Americano with Milk  coffee_name_Cappuccino  coffee_name_Cocoa  \
0                            False             

## Крок 9. Нормалізація та стандартизація

In [18]:
df['money_norm'] = (df['money'] - df['money'].min()) / (df['money'].max() - df['money'].min())
df['money_std'] = (df['money'] - df['money'].mean()) / df['money'].std()

In [19]:
print(df[['money', 'money_norm', 'money_std']].head())
print("Мінімум money_norm:", df['money_norm'].min())
print("Максимум money_norm:", df['money_norm'].max())
print("Середнє money_std:", df['money_std'].mean())
print("Стандартне відхилення money_std:", df['money_std'].std())

   money  money_norm  money_std
0  35.99    0.834554   0.512159
1  35.99    0.834554   0.512159
2  24.00    0.102564  -1.631271
3  38.70    1.000000   0.996620
4  33.80    0.700855   0.120657
Мінімум money_norm: 0.0
Максимум money_norm: 1.0
Середнє money_std: -2.782959048393726e-16
Стандартне відхилення money_std: 1.0


Середнє `money_std` — число порядку 10⁻¹⁶…10⁻¹⁷, тобто фактично 0: це наслідок обмеженої точності обчислень із дробовими числами.

## Крок 10. Збереження очищених даних

In [20]:
final_df = df[['datetime', 'cash_type', 'money', 'coffee_name', 'hour', 'day_of_week', 'month', 'money_norm', 'money_std']]
final_df.to_csv(BASE + 'cleaned_sales.csv', index=False)
print('Збережено:', final_df.shape)

Збережено: (300, 9)


In [21]:
final_df.head()

,datetime,cash_type,money,coffee_name,hour,day_of_week,month,money_norm,money_std
0,2025-03-01 09:27:40.117,card,35.99,Cappuccino,9,5,3,0.834554,0.512159
1,2025-03-01 09:40:11.474,card,35.99,Cocoa,9,5,3,0.834554,0.512159
2,2025-03-01 10:12:26.345,cash,24.00,Espresso,10,5,3,0.102564,-1.631271
3,2025-03-01 10:38:22.609,card,38.70,Latte,10,5,3,1.000000,0.996620
4,2025-03-01 11:33:02.443,card,33.80,Americano with Milk,11,5,3,0.700855,0.120657


### Відтворювана послідовність очищення
Кроки виконано в чіткому порядку: завантаження → пропуски → дублікати → викиди → типи → рядки → нові ознаки → кодування → масштабування → збереження. Кожен крок зафіксовано в коді з коментарями, тому будь-хто може повторити очищення й перевірити його.